# Challenge: evaluate your model with EvalHub

Adapted for the workshop from the EvalHub example
[end_to_end_llm_evaluation_on_openshift.ipynb](https://github.com/opendatahub-io/eval-hub/blob/main/examples/end_to_end_llm_evaluation_on_openshift.ipynb)
(Apache 2.0). You will:

1. Connect to **your own EvalHub** (the one you created in your project) with the `eval-hub-sdk` Python client
2. Submit an evaluation of the workshop model (**Llama 3.2**) on the **arc_easy** benchmark with **lm-evaluation-harness**
3. Find the results in **MLflow**

EvalHub is a Technology Preview in OpenShift AI 3.5. Run the cells from top to bottom.

## Configure the connection

Only one value to fill in: **your login token**. In the OpenShift console, click your user name (top right) →
**Copy login command** → log in → **Display token**, and paste the `sha256~...` token below. Your project and
your EvalHub's address are found automatically.

In [ ]:
import pathlib

# your project (the workbench's project, e.g. <USER_NAME>-canopy)
evalhub_tenant = pathlib.Path("/var/run/secrets/kubernetes.io/serviceaccount/namespace").read_text().strip()

# your EvalHub, reached inside the cluster
evalhub_url = f"https://evalhub.{evalhub_tenant}.svc.cluster.local:8443"

# YOUR login token (console → your name → Copy login command → Display token)
evalhub_token = "sha256~PASTE-YOUR-TOKEN-HERE"

print("project:", evalhub_tenant)
print("EvalHub:", evalhub_url)

## Install the EvalHub SDK

Version 1.0.x matches the EvalHub server in OpenShift AI 3.5.

In [ ]:
%pip install -q "eval-hub-sdk[client]==1.0.7"

## Import the EvalHub SDK

In [ ]:
from evalhub import (
    SyncEvalHubClient,
    JobSubmissionRequest,
    BenchmarkConfig,
    ModelConfig,
    JobStatus,
    ExperimentConfig,
    ExperimentTag,
)
from evalhub.models.api import ModelAuth

## Connect to EvalHub

Create a client and verify the service is reachable.

In [ ]:
client = SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=evalhub_token,
    insecure=True,
    tenant=evalhub_tenant,
)
try:
    # Check health
    health = client.health()
    print(f"✓ EvalHub is healthy: {health['status']}")
    print(f"  Version: {health.get('version', 'unknown')}")
    print(f"  Uptime: {health.get('uptime_seconds', 0):.1f}s")
except Exception as e:
    print(f"✗ Failed to connect: {e}")

## List Available Providers

In [ ]:
providers = client.providers.list()
print(f"✓ Found {len(providers)} providers:")
for provider in providers:
    print(f"  - {provider.resource.id}: {provider.name}")

## List Benchmarks for `lm_evaluation_harness`

Query the available benchmarks under the provider we will use.

In [ ]:
provider_id = "lm_evaluation_harness"
benchmarks = client.benchmarks.list(provider_id=provider_id)
print(f"\n✓ Found {len(benchmarks)} benchmarks for {provider_id}:")
for benchmark in benchmarks:
    print(f"  - {benchmark.id}: {benchmark.name}")

## Configure the model

The workshop model's address and name come from your project's connection **llm-llama32** (the one Labs 4 and 5
use), read with your token. The model needs no API key, but EvalHub expects a Secret with one: `llama32-auth`
(key `api-key`, value `none`), which you created together with your EvalHub.

In [ ]:
import base64, json, ssl, urllib.request

# the model from your project's connection llm-llama32 (Kubernetes API, with your token)
sa = "/var/run/secrets/kubernetes.io/serviceaccount"
req = urllib.request.Request(
    f"https://kubernetes.default.svc/api/v1/namespaces/{evalhub_tenant}/secrets/llm-llama32",
    headers={"Authorization": f"Bearer {evalhub_token}"},
)
conn = json.load(urllib.request.urlopen(req, context=ssl.create_default_context(cafile=f"{sa}/ca.crt")))["data"]

model_url = base64.b64decode(conn["OPENAI_BASE_URL"]).decode()
model_name = base64.b64decode(conn["MODEL_NAME"]).decode()
model_secret = "llama32-auth"
print("model:", model_name, "at", model_url)

## Submit an evaluation job

Run the `arc_easy` benchmark with `lm_evaluation_harness` on 10 examples (the GPU is shared by all groups) and log the results to MLflow.

In [ ]:
# this is the benchmark id, you can find it from the list of benchmarks above
benchmark_id = "arc_easy"

The `lm_evaluation_harness` framework requires a tokenizer. Here we use `google/flan-t5-small` — replace it with a tokenizer appropriate for your model if needed.

In [ ]:
tokenizer_model = "google/flan-t5-small"

In [ ]:
import time

# the job name of your choice (unique per run)
job_name = f"arc-easy-{int(time.time())}"

# the MLflow experiment the results go to
mlflow_experiment_name = "evalhub-challenge"

In [ ]:
job_request = JobSubmissionRequest(
    name=job_name,
    model=ModelConfig(
        url=model_url, name=model_name, auth=ModelAuth(secret_ref=model_secret)
    ),
    benchmarks=[
        BenchmarkConfig(
            id=benchmark_id,
            provider_id=provider_id,
            parameters={
                "num_examples": 10,
                "tokenizer": tokenizer_model,
            },
        )
    ],
    experiment=ExperimentConfig(
        name=mlflow_experiment_name,
        tags=[
            ExperimentTag(key="env", value="workshop"),
        ],
    ),
)

job = client.jobs.submit(job_request)
submitted_job_id = job.id
print(f"✓ Job submitted successfully")
print(f"  Job ID: {submitted_job_id}")
print(f"  State: {job.state}")
print(f"  Created: {job.resource.created_at}")

In [ ]:
# Check status
updated_job = client.jobs.get(submitted_job_id)
print(f"\n✓ Current job state: {updated_job.state}")
if updated_job.status and updated_job.status.message:
    print(f"  Message: {updated_job.status.message.message}")

## Wait for the Job to Complete

Poll the job status until it reaches `COMPLETED` or `FAILED`.

In [ ]:
import time

while updated_job.state not in (JobStatus.COMPLETED, JobStatus.FAILED):
    print(f"\n✓ Current job state: {updated_job.state}")
    time.sleep(5)
    updated_job = client.jobs.get(submitted_job_id)

if updated_job.state == JobStatus.FAILED:
    if updated_job.status and updated_job.status.message:
        print(f"Job failed: {updated_job.status.message.message}")
    else:
        print("Job failed: Unknown")
else:
    print(f"✓ Job completed successfully")

## Inspect the Completed Job

Dump the full job object to see all fields including status, results, experiment metadata, and export details.

In [ ]:
print(updated_job.model_dump_json(indent=2))

### Benchmark Metrics

Extract the metrics returned by the `arc_easy` benchmark.

In [ ]:
print("Benchmark metrics:", updated_job.results.benchmarks[0].metrics)

### MLflow Run ID

The evaluation results are also logged to MLflow. Use the run ID below to find the run in the MLflow UI.

In [ ]:
updated_job.results.benchmarks[0].mlflow_run_id

The run is in MLflow: OpenShift AI dashboard → **Develop & train → Experiments → Launch MLflow**, workspace =
your project, experiment **evalhub-challenge**. Compare runs there if you evaluate again with other settings.

## Clean up (optional)

Deletes the job record in EvalHub (the MLflow run stays) and closes the client.

In [ ]:
# run this cell to delete the job with hard_delete=True
client.jobs.cancel(submitted_job_id, hard_delete=True)

In [ ]:
## Close the client
client.close()